# Практикум 8. Случайный лес

Сопровождает интерактивную лабораторию №8 («Случайный лес»). Тема курса: ансамбли деревьев: бэггинг и устойчивость

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.5, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Одно дерево и лес

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

Xtr_t, Xte_t, ytr, yte = train_test_split(df.text, df.label, test_size=0.3, stratify=df.label, random_state=0)
vec = TfidfVectorizer(max_features=400)
Xtr = vec.fit_transform(Xtr_t); Xte = vec.transform(Xte_t)
tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
rf = RandomForestClassifier(200, oob_score=True, random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("Одно дерево  test:", round(tree.score(Xte, yte), 3))
print("Лес из 200   test:", round(rf.score(Xte, yte), 3), " OOB:", round(rf.oob_score_, 3))

## 2. Точность в зависимости от числа деревьев (тест и OOB)

In [ ]:
ns = [1, 2, 5, 10, 20, 50, 100, 200]
te, oob = [], []
for n in ns:
    m = RandomForestClassifier(n, oob_score=True, random_state=0, n_jobs=-1).fit(Xtr, ytr)
    te.append(m.score(Xte, yte)); oob.append(m.oob_score_ if n >= 5 else np.nan)
plt.plot(ns, te, marker="o", label="тест"); plt.plot(ns, oob, marker="s", label="OOB")
plt.axhline(tree.score(Xte, yte), ls="--", color="gray", label="одно дерево")
plt.xscale("log"); plt.xlabel("число деревьев"); plt.ylabel("accuracy"); plt.legend(); plt.show()

## 3. Что создаёт разнообразие: бутстрэп и случайные признаки

In [ ]:
rows = []
for bs in [True, False]:
    for mf in ["sqrt", None]:
        m = RandomForestClassifier(100, bootstrap=bs, max_features=mf, random_state=0, n_jobs=-1).fit(Xtr, ytr)
        rows.append((bs, mf, round(m.score(Xte, yte), 3)))
pd.DataFrame(rows, columns=["bootstrap", "max_features", "test accuracy"])

In [ ]:
imp = pd.Series(rf.feature_importances_, index=vec.get_feature_names_out()).sort_values(ascending=False).head(15)
imp[::-1].plot.barh(figsize=(6, 5)); plt.title("Важность слов в лесу"); plt.show()

## Задания

1. Измерьте разброс точности одного дерева и леса при разных `random_state` (20 запусков). Что показывает разница?
2. Постройте OOB-ошибку для разной глубины `max_depth`.
3. Почему лес хуже объясним, чем одно дерево? Как использовать важность признаков для частичного объяснения?